# Phase 05 · Bài 01 — Text Processing

**Mục tiêu:** hiểu cách chuỗi văn bản được chuyển thành đơn vị hữu ích cho NLP, phân biệt stemming với lemmatization, và nhận ra lỗi do chuẩn hóa/tiền xử lý không nhất quán.

## 1. Vì sao cần tiền xử lý?

Văn bản là một chuỗi ký tự; phần lớn mô hình cổ điển không tính trực tiếp trên câu chữ. Pipeline cần xác định đơn vị đầu vào, ánh xạ các đơn vị đó thành biểu diễn số và giữ cùng một quy ước khi huấn luyện lẫn suy luận. Một lỗi sớm có thể thay đổi vocabulary và làm tín hiệu của những bước sau bị phân mảnh.

Tiền xử lý không phải lúc nào cũng có nghĩa là loại bỏ thật nhiều thông tin. Viết thường giúp gộp “Apple” và “apple” nhưng có thể làm mất tín hiệu tên riêng. Bỏ dấu câu giúp giảm vocabulary nhưng có thể xóa dấu câu mang cảm xúc hoặc cấu trúc. Mỗi phép biến đổi phải dựa vào tác vụ.

## 2. Tokenization: chọn ranh giới đơn vị

Tokenizer quyết định một câu được chia thành token nào. Các lựa chọn thường gặp là token theo từ, subword, ký tự hoặc byte. Word tokenization phù hợp nhiều baseline cổ điển; subword giúp mô hình xử lý từ hiếm; ký tự/byte có thể bao phủ chuỗi lạ nhưng tạo chuỗi dài hơn.

Tokenizer cần đưa ra quyết định cho dấu câu, chữ viết hoa, số, apostrophe, URL, emoji và từ ghép. Code mẫu dùng biểu thức chính quy gồm các nhánh cho chữ cái, apostrophe bên trong từ, chữ số và dấu câu. Vì chữ số và chữ cái được nhận bằng hai nhánh, “3pm” thành “3”, “pm”. Đó là hành vi của quy tắc đã viết, không phải cách tách duy nhất đúng.

Tokenizer trong bài chỉ nhận A–Z, do đó nó không phải tokenizer tiếng Việt đầy đủ. Tiếng Việt dùng Unicode có dấu và khoảng trắng thường ngăn cách âm tiết chứ không phải lúc nào cũng ngăn cách từ nhiều âm tiết. Muốn làm NLP tiếng Việt nghiêm túc cần tokenizer và tài nguyên phù hợp ngôn ngữ.

## 3. Stemming: cắt hậu tố theo quy tắc

Stemming áp dụng các phép biến đổi hình thức để gom một số biến thể về gần cùng gốc. Phiên bản trong bài triển khai một phần Porter step 1a. Thứ tự điều kiện quan trọng: “sses” và “ies” được xử lý trước; “ss” được giữ nguyên; cuối cùng “s” đơn có thể bị bỏ. Vì vậy “caresses” thành “caress”, còn “ponies” thành “poni”.

Stem không bắt buộc là từ có trong từ điển. Ưu điểm là nhanh, ít tài nguyên và hữu ích khi cần gom biến thể từ trong tìm kiếm thô. Nhược điểm là có thể cắt quá tay (over-stemming) và gộp hai từ khác nghĩa; nếu cắt chưa đủ thì các biến thể vẫn không được gom (under-stemming).

## 4. Lemmatization: cần hình thái và từ loại

Lemma là dạng từ điển của một từ. Việc tìm lemma cần biết biến tố và thường cần POS: “running” là động từ thì có thể về “run”; “better” là tính từ so sánh thì có thể về “good”; “were” là dạng quá khứ của “be”. Cùng một chuỗi có thể có cách phân tích khác nhau tùy vai trò ngữ pháp.

Bài dùng bảng tra khóa theo cặp (từ thường, POS), kèm fallback đơn giản cho “-ing” và danh từ số nhiều. Điều này dạy được cấu trúc nhưng không bao quát tiếng Anh: “watched” không được đổi vì bảng/fallback chưa xử lý “-ed”; POS tagger toy cũng gán phần lớn token thành NOUN. Lemmatizer sản xuất cần từ điển hình thái hoặc mô hình/ngôn ngữ phù hợp.

| Từ đầu vào | Stem step 1a | Lemma theo ngữ cảnh |
|---|---|---|
| running (VERB) | running | run |
| ponies (NOUN) | poni | pony |
| better (ADJ) | better | good |

## 5. Đưa vào pipeline và kiểm soát lỗi

Các bước tokenization, chuyển chữ thường, POS tagging, stemming/lemmatization phải được lưu thành một pipeline nhất quán. Nếu training bỏ stopword hoặc lemmatize nhưng inference dùng văn bản thô, model nhận phân phối feature khác. Nên đóng gói preprocessing cùng model và cố định phiên bản thư viện; thay đổi tokenizer có thể làm kết quả trôi mà không phát sinh lỗi chương trình.

Notebook chạy code mẫu của repo rồi thử ba câu mới. Quan sát chính là cùng một từ có thể có ba kết quả khác nhau: token gốc, stem theo luật, lemma dựa vào POS/bảng từ. Những trường hợp lỗi cũng hữu ích vì chúng cho thấy chính xác phạm vi của implementation toy.

## A. Chạy code mẫu của bài

Cell này gọi code/main.py gốc và ghi stdout/exit code vào notebook.

In [1]:
from pathlib import Path
import subprocess
import sys

PHASE = "phases/05-nlp-foundations-to-advanced"
LESSON = "01-text-processing"
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / PHASE / "README.md").is_file():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Mở Jupyter từ thư mục repo hoặc thư mục con của repo.")
script = REPO / PHASE / LESSON / "code" / "main.py"
result = subprocess.run([sys.executable, str(script)], cwd=REPO, text=True, capture_output=True)
print("Bài:", "Text Processing")
print("Lệnh:", script.relative_to(REPO))
print("Python:", sys.version.split()[0])
print("Exit code:", result.returncode)
print("\n--- stdout ---")
print(result.stdout.rstrip())
if result.stderr:
    print("\n--- stderr ---")
    print(result.stderr.rstrip())
if result.returncode != 0:
    raise RuntimeError(f"Code mẫu của {LESSON} thoát với mã {result.returncode}")


Bài: Text Processing
Lệnh: phases\05-nlp-foundations-to-advanced\01-text-processing\code\main.py
Python: 3.12.10
Exit code: 0

--- stdout ---
input:  The cats were running at 3pm.
tokens: ['The', 'cats', 'were', 'running', 'at', '3', 'pm', '.']
stems:  ['the', 'cat', 'were', 'running', 'at', '3', 'pm', '.']
lemmas: ['the', 'cat', 'be', 'run', 'at', '3', 'pm', '.']


## B. Thực hành riêng — quan sát luật trên câu mới

Ta dùng cùng ý tưởng của bài nhưng thay câu đầu vào. Bộ POS ở đây cố ý nhỏ để thấy lemmatizer phụ thuộc vào tag; nó không phải POS tagger tổng quát.

In [2]:
import re

WORD_RE = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?|[0-9]+|[^\sA-Za-z0-9]")

def tokenize(text):
    return WORD_RE.findall(text)

def stem_step_1a(word):
    word = word.lower()
    if word.endswith("sses"):
        return word[:-2]
    if word.endswith("ies"):
        return word[:-2]
    if word.endswith("ss"):
        return word
    if word.endswith("s") and len(word) > 1:
        return word[:-1]
    return word

LEMMAS = {
    ("running", "VERB"): "run", ("ran", "VERB"): "run",
    ("were", "VERB"): "be", ("was", "VERB"): "be",
    ("better", "ADJ"): "good", ("cats", "NOUN"): "cat", ("ponies", "NOUN"): "pony",
}

def lemmatize(word, pos):
    key = (word.lower(), pos)
    if key in LEMMAS:
        return LEMMAS[key]
    if pos == "VERB" and word.lower().endswith("ing"):
        return word[:-3].lower()
    if pos == "NOUN" and word.lower().endswith("s"):
        return word[:-1].lower()
    return word.lower()

def tiny_pos_tagger(tokens):
    verbs = {"were", "was", "running", "ran", "runs"}
    adjectives = {"better", "best"}
    return [(word, "VERB" if word.lower() in verbs else
             "ADJ" if word.lower() in adjectives else "NOUN")
            for word in tokens]

examples = [
    "The cats were running at 3pm.",
    "The ponies were better than the cats.",
    "I can't say the watched results were good!",
]
for text in examples:
    tokens = tokenize(text)
    tags = tiny_pos_tagger(tokens)
    stems = [stem_step_1a(token) for token in tokens]
    lemmas = [lemmatize(word, pos) for word, pos in tags]
    print("text:  ", text)
    print("tokens:", tokens)
    print("stems: ", stems)
    print("lemmas:", lemmas)
    print()

print("Stemming cho ponies → poni; lemma-table cho ponies → pony.")
print("watched không đổi vì bảng lemma/fallback chưa mô tả hậu tố -ed.")

text:   The cats were running at 3pm.
tokens: ['The', 'cats', 'were', 'running', 'at', '3', 'pm', '.']
stems:  ['the', 'cat', 'were', 'running', 'at', '3', 'pm', '.']
lemmas: ['the', 'cat', 'be', 'run', 'at', '3', 'pm', '.']

text:   The ponies were better than the cats.
tokens: ['The', 'ponies', 'were', 'better', 'than', 'the', 'cats', '.']
stems:  ['the', 'poni', 'were', 'better', 'than', 'the', 'cat', '.']
lemmas: ['the', 'pony', 'be', 'good', 'than', 'the', 'cat', '.']

text:   I can't say the watched results were good!
tokens: ['I', "can't", 'say', 'the', 'watched', 'results', 'were', 'good', '!']
stems:  ['i', "can't", 'say', 'the', 'watched', 'result', 'were', 'good', '!']
lemmas: ['i', "can't", 'say', 'the', 'watched', 'result', 'be', 'good', '!']

Stemming cho ponies → poni; lemma-table cho ponies → pony.
watched không đổi vì bảng lemma/fallback chưa mô tả hậu tố -ed.


## C. Đọc kết quả và tự kiểm tra

1. Vì sao ran thành run cần bảng lemma thay vì chỉ cắt hậu tố?
2. Vì sao ponies thành poni là stemming chứ chưa phải lemmatization?
3. Regex ở đây chỉ nhận chữ cái Latin A–Z. Với tiếng Việt có dấu, tokenizer đó có thể làm gì sai?
4. Nếu train đã lower-case và lemmatize, inference cần giữ điều gì giống hệt?

**Giới hạn:** đây là tokenizer/lemmatizer minh họa bằng luật, không thay thế spaCy, NLTK hay bộ phân tích hình thái đa ngôn ngữ.